# Customer Churn Analysis
**Author:** Dipak Ingale  
**Project:** Churn Analysis — EDA & Insights  
**Dataset:** Customer_Churn.csv  
**Date:** 2026

---

## Objective
Perform Exploratory Data Analysis (EDA) on the customer churn dataset to:
- Understand the distribution of churned vs retained customers
- Identify key drivers of churn (plan type, contract type, charges, CSAT, etc.)
- Visualize patterns and relationships in the data
- Derive actionable business insights

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

print('Libraries loaded successfully.')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('Customer_Churn.csv')
print(f'Dataset Shape: {df.shape}')
df.head()

## 3. Data Overview

In [ ]:
print('=== Column Data Types ===')
print(df.dtypes)
print()
print('=== Basic Info ===')
df.info()

In [ ]:
print('=== Statistical Summary (Numeric) ===')
df.describe()

In [ ]:
print('=== Statistical Summary (Categorical) ===')
df.describe(include='object')

## 4. Data Cleaning & Preprocessing

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct.round(2)})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print(missing_df)

In [ ]:
# Convert date columns
date_cols = ['subscription_start_date', 'renewal_date', 'cancellation_date', 'complaint_date', 'dob']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Standardize text columns
df['subscription_type'] = df['subscription_type'].str.strip().str.title()
df['plan_type'] = df['plan_type'].str.strip().str.title()
df['contract_type'] = df['contract_type'].str.strip().str.title()
df['gender'] = df['gender'].str.strip().str.title()
df['country'] = df['country'].str.strip().str.title()
df['state'] = df['state'].str.strip().str.title()
df['cancellation_reason'] = df['cancellation_reason'].str.strip()
df['escalations'] = df['escalations'].str.strip().str.upper()

# Calculate customer tenure in months
reference_date = pd.Timestamp('2025-01-01')
df['tenure_months'] = ((reference_date - df['subscription_start_date']).dt.days / 30).round(1)

# Calculate age from dob
df['age'] = ((reference_date - df['dob']).dt.days / 365.25).round(0).astype('Int64')

# Age group
bins = [0, 25, 35, 45, 55, 100]
labels = ['<25', '25-34', '35-44', '45-54', '55+']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)

# Flag for has_complaint
df['has_complaint'] = df['complaint_date'].notna().astype(int)

# Monthly charges category
df['charge_band'] = pd.cut(df['monthly_charges'],
                            bins=[0, 10, 15, 20, 100],
                            labels=['Low (<10)', 'Mid (10-15)', 'High (15-20)', 'Premium (>20)'])

print('Preprocessing complete.')
df[['tenure_months', 'age', 'age_group', 'has_complaint', 'charge_band']].head()

## 5. Churn Overview

In [ ]:
churn_counts = df['churn_flag'].value_counts()
churn_rate = df['churn_flag'].mean() * 100

print(f'Total Customers     : {len(df)}')
print(f'Churned Customers   : {churn_counts.get(1, 0)}')
print(f'Retained Customers  : {churn_counts.get(0, 0)}')
print(f'Overall Churn Rate  : {churn_rate:.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
labels_map = {0: 'Retained', 1: 'Churned'}
colors = ['#4CAF50', '#F44336']
axes[0].bar([labels_map[k] for k in churn_counts.index], churn_counts.values, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Churn vs Retained — Count')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(churn_counts.values,
            labels=[labels_map[k] for k in churn_counts.index],
            autopct='%1.1f%%', colors=colors, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn Distribution')

plt.tight_layout()
plt.savefig('churn_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Churn by Subscription & Contract Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Churn rate by subscription type
sub_churn = df.groupby('subscription_type')['churn_flag'].mean().sort_values(ascending=False) * 100
sub_churn.plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2', len(sub_churn)), edgecolor='white', rot=0)
axes[0].set_title('Churn Rate by Subscription Type')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_xlabel('Subscription Type')
for i, v in enumerate(sub_churn.values):
    axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

# Churn rate by contract type
con_churn = df.groupby('contract_type')['churn_flag'].mean().sort_values(ascending=False) * 100
con_churn.plot(kind='bar', ax=axes[1], color=['#E57373', '#81C784'], edgecolor='white', rot=0)
axes[1].set_title('Churn Rate by Contract Type')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_xlabel('Contract Type')
for i, v in enumerate(con_churn.values):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('churn_by_subscription_contract.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Churn by Plan Type

In [ ]:
plan_churn = df.groupby('plan_type').agg(
    total=('churn_flag', 'count'),
    churned=('churn_flag', 'sum')
).assign(churn_rate=lambda x: (x['churned'] / x['total'] * 100).round(2))
print(plan_churn)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plan_churn['churn_rate'].sort_values(ascending=False).plot(
    kind='bar', ax=axes[0], color=sns.color_palette('RdYlGn_r', 3), edgecolor='white', rot=0)
axes[0].set_title('Churn Rate by Plan Type (%)')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_xlabel('Plan Type')
for i, v in enumerate(plan_churn['churn_rate'].sort_values(ascending=False).values):
    axes[0].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

# Stacked bar — churned vs retained by plan
plan_stack = df.groupby(['plan_type', 'churn_flag']).size().unstack(fill_value=0)
plan_stack.columns = ['Retained', 'Churned']
plan_stack.plot(kind='bar', ax=axes[1], color=['#4CAF50', '#F44336'], edgecolor='white', rot=0)
axes[1].set_title('Churned vs Retained by Plan Type')
axes[1].set_ylabel('Customer Count')
axes[1].set_xlabel('Plan Type')
axes[1].legend(title='Status')

plt.tight_layout()
plt.savefig('churn_by_plan.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Churn Score & CLTV Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Churn Score Distribution by Churn Flag
for flag, label, color in [(0, 'Retained', '#4CAF50'), (1, 'Churned', '#F44336')]:
    axes[0].hist(df[df['churn_flag'] == flag]['churn_score'],
                 bins=20, alpha=0.6, label=label, color=color, edgecolor='white')
axes[0].set_title('Churn Score Distribution')
axes[0].set_xlabel('Churn Score')
axes[0].set_ylabel('Count')
axes[0].legend()

# CLTV Distribution by Churn Flag
for flag, label, color in [(0, 'Retained', '#4CAF50'), (1, 'Churned', '#F44336')]:
    axes[1].hist(df[df['churn_flag'] == flag]['cltv'],
                 bins=20, alpha=0.6, label=label, color=color, edgecolor='white')
axes[1].set_title('CLTV Distribution by Churn Status')
axes[1].set_xlabel('Customer Lifetime Value (CLTV)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('churn_score_cltv.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary stats
print('=== Churn Score Summary ===')
print(df.groupby('churn_flag')['churn_score'].describe().round(2))
print()
print('=== CLTV Summary ===')
print(df.groupby('churn_flag')['cltv'].describe().round(2))

## 9. Monthly Charges Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot of monthly charges by churn
df['churn_label'] = df['churn_flag'].map({0: 'Retained', 1: 'Churned'})
sns.boxplot(data=df, x='churn_label', y='monthly_charges',
            palette={'Retained': '#4CAF50', 'Churned': '#F44336'}, ax=axes[0])
axes[0].set_title('Monthly Charges by Churn Status')
axes[0].set_xlabel('Churn Status')
axes[0].set_ylabel('Monthly Charges (INR)')

# Churn rate by charge band
charge_churn = df.groupby('charge_band', observed=True)['churn_flag'].mean() * 100
charge_churn.plot(kind='bar', ax=axes[1], color=sns.color_palette('YlOrRd', len(charge_churn)), edgecolor='white', rot=0)
axes[1].set_title('Churn Rate by Monthly Charge Band')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_xlabel('Charge Band')
for i, v in enumerate(charge_churn.values):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('monthly_charges_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Cancellation Reason Analysis

In [ ]:
churned_df = df[df['churn_flag'] == 1].copy()
reason_counts = churned_df['cancellation_reason'].value_counts()
print('Top Cancellation Reasons:')
print(reason_counts)

plt.figure(figsize=(10, 5))
colors = sns.color_palette('Reds_r', len(reason_counts))
bars = plt.barh(reason_counts.index, reason_counts.values, color=colors, edgecolor='white')
plt.title('Cancellation Reasons (Churned Customers)')
plt.xlabel('Number of Customers')
for bar, val in zip(bars, reason_counts.values):
    plt.text(val + 0.1, bar.get_y() + bar.get_height() / 2,
             str(val), va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('cancellation_reasons.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. CSAT Score & Complaint Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CSAT vs Churn
csat_df = df.dropna(subset=['csat_score'])
sns.boxplot(data=csat_df, x='churn_label', y='csat_score',
            palette={'Retained': '#4CAF50', 'Churned': '#F44336'}, ax=axes[0])
axes[0].set_title('CSAT Score by Churn Status')
axes[0].set_xlabel('Churn Status')
axes[0].set_ylabel('CSAT Score')

# Has complaint vs churn rate
complaint_churn = df.groupby('has_complaint')['churn_flag'].mean() * 100
complaint_churn.index = complaint_churn.index.map({0: 'No Complaint', 1: 'Has Complaint'})
complaint_churn.plot(kind='bar', ax=axes[1], color=['#81C784', '#E57373'], edgecolor='white', rot=0)
axes[1].set_title('Churn Rate: Complaint vs No Complaint')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_xlabel('')
for i, v in enumerate(complaint_churn.values):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('csat_complaint_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== CSAT Score Summary ===')
print(csat_df.groupby('churn_flag')['csat_score'].describe().round(2))

## 12. Demographic Analysis (Gender & Age)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Churn rate by gender
gender_churn = df.groupby('gender')['churn_flag'].mean() * 100
gender_churn.plot(kind='bar', ax=axes[0], color=['#E91E63', '#2196F3'], edgecolor='white', rot=0)
axes[0].set_title('Churn Rate by Gender')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_xlabel('Gender')
for i, v in enumerate(gender_churn.values):
    axes[0].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')

# Churn rate by age group
age_churn = df.groupby('age_group', observed=True)['churn_flag'].mean() * 100
age_churn.plot(kind='bar', ax=axes[1], color=sns.color_palette('coolwarm', len(age_churn)), edgecolor='white', rot=0)
axes[1].set_title('Churn Rate by Age Group')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_xlabel('Age Group')
for i, v in enumerate(age_churn.values):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('demographic_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Geographic Analysis (Country & State)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Churn rate by top states
state_churn = df.groupby('state').agg(
    total=('churn_flag', 'count'),
    churned=('churn_flag', 'sum')
).assign(churn_rate=lambda x: (x.churned / x.total * 100).round(1))
state_churn_top = state_churn[state_churn['total'] >= 3].sort_values('churn_rate', ascending=False).head(10)

state_churn_top['churn_rate'].plot(
    kind='barh', ax=axes[0],
    color=sns.color_palette('RdYlGn_r', len(state_churn_top)), edgecolor='white')
axes[0].set_title('Top States by Churn Rate')
axes[0].set_xlabel('Churn Rate (%)')
axes[0].set_ylabel('State')

# Customer count by country
country_counts = df.groupby(['country', 'churn_label']).size().unstack(fill_value=0)
country_counts.plot(kind='bar', ax=axes[1], color=['#4CAF50', '#F44336'], edgecolor='white', rot=0)
axes[1].set_title('Customers by Country')
axes[1].set_ylabel('Count')
axes[1].set_xlabel('Country')
axes[1].legend(title='Status')

plt.tight_layout()
plt.savefig('geographic_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Tenure Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tenure distribution by churn
for flag, label, color in [(0, 'Retained', '#4CAF50'), (1, 'Churned', '#F44336')]:
    axes[0].hist(df[df['churn_flag'] == flag]['tenure_months'],
                 bins=15, alpha=0.6, label=label, color=color, edgecolor='white')
axes[0].set_title('Tenure Distribution by Churn Status')
axes[0].set_xlabel('Tenure (Months)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Tenure vs churn score scatter
scatter_colors = df['churn_flag'].map({0: '#4CAF50', 1: '#F44336'})
axes[1].scatter(df['tenure_months'], df['churn_score'], c=scatter_colors, alpha=0.6, edgecolors='white')
axes[1].set_title('Tenure vs Churn Score')
axes[1].set_xlabel('Tenure (Months)')
axes[1].set_ylabel('Churn Score')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#4CAF50', label='Retained'), Patch(facecolor='#F44336', label='Churned')]
axes[1].legend(handles=legend_elements)

plt.tight_layout()
plt.savefig('tenure_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== Tenure Summary ===')
print(df.groupby('churn_flag')['tenure_months'].describe().round(2))

## 15. Correlation Heatmap

In [ ]:
numeric_cols = ['monthly_charges', 'cltv', 'churn_score', 'churn_flag',
                'tenure_months', 'age', 'has_complaint', 'csat_score', 'complaint_count']
corr_df = df[numeric_cols].dropna()
corr_matrix = corr_df.corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 9})
plt.title('Correlation Heatmap — Numeric Features', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 16. Escalation Impact on Churn

In [ ]:
esc_df = df.dropna(subset=['escalations'])
if len(esc_df) > 0:
    esc_churn = esc_df.groupby('escalations')['churn_flag'].mean() * 100
    esc_count = esc_df.groupby('escalations')['churn_flag'].count()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    esc_churn.plot(kind='bar', ax=axes[0], color=['#81C784', '#E57373'], edgecolor='white', rot=0)
    axes[0].set_title('Churn Rate by Escalation (Y/N)')
    axes[0].set_ylabel('Churn Rate (%)')
    axes[0].set_xlabel('Escalated?')
    for i, v in enumerate(esc_churn.values):
        axes[0].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')

    esc_count.plot(kind='bar', ax=axes[1], color=['#81C784', '#E57373'], edgecolor='white', rot=0)
    axes[1].set_title('Customer Count by Escalation Status')
    axes[1].set_ylabel('Count')
    axes[1].set_xlabel('Escalated?')

    plt.tight_layout()
    plt.savefig('escalation_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No escalation data available.')

## 17. Key Insights Summary

In [ ]:
print('=' * 60)
print('         CUSTOMER CHURN ANALYSIS — KEY INSIGHTS')
print('=' * 60)

total = len(df)
churned = df['churn_flag'].sum()
churn_rate = churned / total * 100

print(f'\n1. OVERALL CHURN RATE: {churn_rate:.1f}% ({churned} of {total} customers)')

top_reason = df[df['churn_flag'] == 1]['cancellation_reason'].value_counts().idxmax()
print(f'\n2. TOP CANCELLATION REASON: {top_reason}')

highest_churn_plan = df.groupby('plan_type')['churn_flag'].mean().idxmax()
plan_rate = df.groupby('plan_type')['churn_flag'].mean().max() * 100
print(f'\n3. HIGHEST CHURN BY PLAN: {highest_churn_plan} ({plan_rate:.1f}%)')

monthly_rate = df[df['contract_type'] == 'Monthly']['churn_flag'].mean() * 100
annual_rate = df[df['contract_type'] == 'Annual']['churn_flag'].mean() * 100
print(f'\n4. CONTRACT TYPE IMPACT:')
print(f'   Monthly Contract Churn : {monthly_rate:.1f}%')
print(f'   Annual Contract Churn  : {annual_rate:.1f}%')

avg_score_churned = df[df['churn_flag'] == 1]['churn_score'].mean()
avg_score_retained = df[df['churn_flag'] == 0]['churn_score'].mean()
print(f'\n5. CHURN SCORE:')
print(f'   Avg Churn Score (Churned)  : {avg_score_churned:.1f}')
print(f'   Avg Churn Score (Retained) : {avg_score_retained:.1f}')

avg_cltv_churned = df[df['churn_flag'] == 1]['cltv'].mean()
avg_cltv_retained = df[df['churn_flag'] == 0]['cltv'].mean()
print(f'\n6. AVERAGE CLTV:')
print(f'   Churned Customers  : {avg_cltv_churned:.0f}')
print(f'   Retained Customers : {avg_cltv_retained:.0f}')

complaint_churn_rate = df[df['has_complaint'] == 1]['churn_flag'].mean() * 100
no_complaint_churn_rate = df[df['has_complaint'] == 0]['churn_flag'].mean() * 100
print(f'\n7. COMPLAINT IMPACT:')
print(f'   With Complaint    : {complaint_churn_rate:.1f}% churn rate')
print(f'   Without Complaint : {no_complaint_churn_rate:.1f}% churn rate')

print('\n' + '=' * 60)

## 18. Recommendations

Based on the EDA findings, the following recommendations are proposed:

| # | Finding | Recommendation |
|---|---------|----------------|
| 1 | Monthly contract customers churn significantly more than annual | Incentivize monthly-plan users to upgrade to annual contracts via discounts |
| 2 | Top reason: 'Switched to competitor' / 'Too expensive' | Introduce competitive pricing tiers and retention offers |
| 3 | Customers with high churn scores (>70) are almost always churned | Deploy proactive retention outreach when churn score crosses 65 |
| 4 | Customers with complaints have much higher churn rates | Improve customer support SLA and complaint resolution time |
| 5 | Low CSAT scores correlate with churn | Implement post-complaint satisfaction surveys and quick-win fixes |
| 6 | Shorter-tenure customers are more prone to churn | Launch onboarding drip campaigns for new users in first 3 months |
| 7 | Referred customers show distinct churn patterns | Segment churn prevention campaigns by acquisition channel |